# 06 Random Forest Modelling

This notebook trains and evaluates a Random Forest classifier using the current Dublin Bikes feature set and the current `band_30min` target definition.

In [ ]:
import torch
import time


print("torch:", torch.__version__)
print("cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("archs:", torch.cuda.get_arch_list())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

x = torch.rand(3000, 3000, device=device)
y = torch.rand(3000, 3000, device=device)

torch.cuda.synchronize()
t0 = time.time()
z = x @ y
torch.cuda.synchronize()
t1 = time.time()

print("Result device:", z.device)
print("Elapsed:", t1 - t0, "seconds")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED) 
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

PROJECT_ROOT = Path("..").resolve()
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "dublin_bikes_features_with_trend.parquet"
if not FEATURES_PATH.exists():
    FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "dublin_bikes_features_with_trend.csv.gz"

if FEATURES_PATH.suffix == ".parquet":
    df = pd.read_parquet(FEATURES_PATH).copy()
else:
    df = pd.read_csv(FEATURES_PATH).copy()

df["last_reported"] = pd.to_datetime(df["last_reported"])
df = df.sort_values(["station_id", "last_reported"]).copy()

print("Loaded:", FEATURES_PATH)
print(df.shape)
df.head()


## Target creation

In [ ]:

# Create the 30-minute-ahead target using the next available observation at or after t + 30 min
def get_target(group):
    group = group.sort_values("last_reported").copy()
    sid = group["station_id"].iloc[0]

    times = group["last_reported"].to_numpy()
    values = group["utilisation"].to_numpy()

    target = np.full(len(group), np.nan)

    for i in range(len(times)):
        t_future = times[i] + np.timedelta64(30, "m")
        idx = np.searchsorted(times, t_future, side="left")
        if idx < len(times):
            target[i] = values[idx]

    group["utilisation_tplus_30"] = target
    group["station_id"] = sid
    return group

parts = []
for _, g in df.groupby("station_id", sort=False):
    parts.append(get_target(g))

df = pd.concat(parts, ignore_index=True)

def utilisation_band(u):
    if u < 0.05:
        return 0
    elif u < 0.30:
        return 1
    elif u < 0.70:
        return 2
    elif u < 0.95:
        return 3
    else:
        return 4

df["band_30min"] = df["utilisation_tplus_30"].apply(utilisation_band)
band_labels = [0, 1, 2, 3, 4]

print("Band distribution:")
print(df["band_30min"].value_counts(normalize=True).sort_index())


## Cleaning

In [ ]:

df = df.dropna(subset=["utilisation_tplus_30", "util_slope", "util_vs_mean"]).copy()
print(df.shape)


## Features and time-based split

In [ ]:

features = [
    "utilisation",
    "roll_std_30min",
    "hour_slot",
    "is_weekday",
    "center_index",
    "area_type_str",
    "util_slope",
    "util_vs_mean",
]

X = df[features].copy()
X = pd.get_dummies(X, columns=["area_type_str"], drop_first=True)

cutoff = df["last_reported"].max() - pd.Timedelta(days=120)
train_mask = df["last_reported"] < cutoff

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = df["band_30min"][train_mask], df["band_30min"][~train_mask]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).sort_index())


## Random Forest model

In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)
print("Training complete.")


## Evaluation

In [ ]:

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification report:")
print(classification_report(y_test, y_pred_rf, labels=band_labels))


In [ ]:

cm = confusion_matrix(y_test, y_pred_rf, labels=band_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=band_labels,
    yticklabels=band_labels
)
plt.title("Random Forest — Confusion Matrix")
plt.ylabel("True Band")
plt.xlabel("Predicted Band")
plt.tight_layout()
plt.show()


In [ ]:

importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importances.sort_values().plot(kind="barh", figsize=(8, 6), color="steelblue")
plt.title("Random Forest — Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()
